# Train Gaussian Process Emulators

In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

import fates_calibration_library.utils as utils
import fates_calibration_library.clm_functions as clm
import fates_calibration_library.emulator_functions as em

import tensorflow as tf
import gpflow

import matplotlib.pyplot as plt
import importlib

2025-06-26 10:30:02.920192: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-26 10:30:04.091390: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-26 10:30:05.983085: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-26 10:30:05.983998: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-26 10:30:21.459306: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

## Set Up
Load files, set up ensemble information

In [2]:
# dataset with land area 
land_frac_ds_file = os.path.join("/glade/derecho/scratch/afoster/archive",
                            "ctsm60SP_bigleaf_fullgrid/lnd/hist",
                            "ctsm60SP_bigleaf_fullgrid.clm2.h0.0001-02-01-00000.nc")

obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'
obs_config = utils.get_config_file(obs_config_file)

# directories
mesh_dir = '/glade/work/afoster/FATES_calibration/mesh_files'
hist_dir = '/glade/work/afoster/FATES_calibration/history_files/compiled_files'
emulator_dir = '/glade/work/afoster/FATES_calibration/emulators'
fig_dir = '/glade/work/afoster/FATES_calibration/figures'
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'

# grab pft names
default_param = xr.open_dataset(os.path.join(param_dir,
                                             'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'))
all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]
default_norm = pd.read_csv(os.path.join(param_dir, 'normalized_parameters.csv'), index_col=[0])

# variables to emulate
calibration_vars = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF']

# test/train split
n_test = 50

In [3]:
# information about each ensemble
ens_dict = {'dompft':
            {'mesh_file': os.path.join(mesh_dir, 'dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'dominant_grid.nc'),
             'ensemble_file': os.path.join(hist_dir, 'fates_dompft_annual_means.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh', 'fates_lh_key.csv'),
             'pfts': [1, 2, 3, 12, 13, 14],
             'obs_df': os.path.join(mesh_dir, 'dominant_grid.csv'),
            }
           }

In [4]:
# choose ensemble
ensemble = 'dompft'

### Load Latin Hypercube Key

In [5]:
lhc_key = pd.read_csv(ens_dict[ensemble]['lhc_key_file'], index_col=[0])
lhc_key = lhc_key.drop(columns=['ensemble'])
param_names = lhc_key.columns
num_params = len(param_names)

In [6]:
obs = pd.read_csv(ens_dict[ensemble]['obs_df'], index_col=[0])

### Load Dictionary of Potential Kernels

In [8]:
gp_kernels = em.build_kernel_dict(num_params)

## Train Emulators
Loop through each pft and variable to emulate and train and save emulators

In [9]:
variable = 'GPP'
pft = 1
pft_name = all_pfts[pft-1]

In [10]:
default_pft = default_norm[default_norm.pft == pft]
default_pft = default_pft.drop(columns = ['pft'])

# get observations for this pft
obs_pft = obs[obs.pft == pft_name]

In [11]:
obs_mean, obs_sd = em.get_obs_mean_and_sd(obs_pft, obs_config[variable]['var'])

In [ ]:
# get the gridcells for this PFT
pft_grid = clm.get_pft_grids(ens_dict[ensemble]['land_mask_file'],
                         ens_dict[ensemble]['mesh_file'], pft)
# subset the ensemble for just this pft
pft_ens = clm.get_pft_ensemble(ens_dict[ensemble]['ensemble_file'],
                               pft_grid, land_frac_ds_file)

In [ ]:
pft_mean = clm.weighted_mean(pft_ens, variable)
X_test, X_train, y_test, y_train = em.split_dataset(pft_mean, lhc_key, n_test)

In [ ]:
kernel = em.select_kernel(gp_kernels, X_train, X_test, y_train, y_test,
                          variable, verbose=True)

In [ ]:
opt_logs, model = em.train_emulator(X_train, y_train, kernel)

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

In [ ]:
test_df = em.test_emulator(model, X_test, y_test, variable)
r2 = r2_score(test_df[f"{variable}_test"], test_df[f"{variable}_pred"])
rmse = root_mean_squared_error(test_df[f"{variable}_test"], test_df[f"{variable}_pred"])
em_sd = np.mean(test_df[f"{variable}_sd"].values)

In [ ]:
em.plot_emulator_validation(test_df, variable, r2, rmse)

In [ ]:
import importlib
importlib.reload(em)

In [ ]:
dat = em.plot_oaat_sens(param_names, model, default_pft)

In [ ]:
sens_df = em.sensitivity_analysis(model, param_names)

In [ ]:
em.plot_global_sensitivity(sens_df, variable)

In [ ]:
from esem.utils import get_random_params

In [ ]:
sample = get_random_params(len(param_names), 10000)

In [ ]:
pred_sampled, pred_sampled_var = model.predict_y(sample)

In [ ]:
plot_emulated_sample(pred_sampled.numpy().flatten(), obs_mean, obs_sd, 'BETT', variable, '')

In [ ]:
r2_dict = {}
sd_dict = {}
for pft in ens_dict[ensemble]['pfts']:

    pft_name = all_pfts[pft-1]

    r2_dict[pft_name] = {}
    sd_dict[pft_name] = {}

    # get the gridcells for this PFT
    pft_grid = clm.get_pft_grids(ens_dict[ensemble]['land_mask_file'],
                             ens_dict[ensemble]['mesh_file'], pft)

    # subset the ensemble for just this pft
    pft_ens = clm.get_pft_ensemble(ens_dict[ensemble]['ensemble_file'],
                                   pft_grid, land_frac_ds_file)

    for variable in calibration_vars:
        # calculate area-weighted mean for this variable
        pft_mean = clm.weighted_mean(pft_ens, variable)

        # split into testing and training datasets
        X_test, X_train, y_test, y_train = em.split_dataset(pft_mean, lhc_key,
                                                            n_test)

        # select best kernel for gp emulator
        kernel = em.select_kernel(gp_kernels, X_train, X_test, y_train, y_test)

        # save to file
        r2, sd = em.train_val_save(X_train, X_test, y_train, y_test, kernel,
                                   out_file=os.path.join(fig_dir, f"{pft_name}_{variable}_emulator_validation.png"),
                                   save_dir=os.path.join(emulator_dir, f"{pft_name}_{variable}"))

        r2_dict[pft_name][f'r2_{variable}'] = r2
        r2_dict[pft_name][f'sd_{variable}'] = sd